# Kvasir-VQA x1 — Late fusion ensemble (text + image)

Ensemble the text-only TF-IDF + Logistic Regression model and the image-only frozen ViT + Logistic Regression model using:
- Weighted averaging of class probabilities (weight tuned on validation).
- A simple stacking meta-classifier trained on validation probabilities.

Outputs are saved under `2_modeling/03_fusion/out/02_late_ensemble/` following the existing x1 conventions.


In [1]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoImageProcessor, ViTModel

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import StandardScaler

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


2026-01-27 03:21:16.266085: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-27 03:21:16.266131: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-27 03:21:16.267435: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-27 03:21:16.274428: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-27 03:21:17.186101: W tensorflow/compiler/tf2

In [2]:
# Paths & config

def find_dataset_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "0_dataset_prep").exists():
            return p
    raise RuntimeError(f"Could not locate dataset root containing '0_dataset_prep'. cwd={start}")

DATA_ROOT = find_dataset_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
OUT_DIR = DATA_ROOT / "2_modeling" / "03_fusion" / "out" / "02_late_ensemble"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "google/vit-base-patch16-224-in21k"
BATCH_SIZE = 16
NUM_WORKERS = 0
TOP_K = 200

MAX_FEATURES = 5000
NGRAM_RANGE = (1, 2)
WEIGHT_GRID = np.linspace(0.0, 1.0, 21)  # 0.00, 0.05, ..., 1.00

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TORCH_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

print("Data root:", DATA_ROOT)
print("Metadata:", META_CSV)
print("Out dir:", OUT_DIR)
print("Device:", DEVICE)
print("Torch dtype:", TORCH_DTYPE)


Data root: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Metadata: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/metadata/metadata_enriched.csv
Out dir: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/03_fusion/out/02_late_ensemble
Device: cuda
Torch dtype: torch.float16


In [3]:
# Load metadata and splits
meta = pd.read_csv(META_CSV)

images_base = DATA_ROOT / "0_dataset_prep"
meta["image_path"] = meta["image_path"].apply(
    lambda p: str((images_base / p).resolve()) if not Path(p).is_absolute() else p
)

meta["question_norm"] = meta["question"].fillna("").astype(str).str.lower().str.strip()
meta["answer_norm"] = meta["answer"].fillna("").astype(str).str.lower().str.strip()

if "split" not in meta.columns:
    raise RuntimeError("Missing 'split' column. Run dataset prep split step first.")

train_df = meta[meta["split"] == "train"].reset_index(drop=True)
val_df = meta[meta["split"] == "validation"].reset_index(drop=True)
test_df = meta[meta["split"] == "test"].reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})


{'train': 46966, 'val': 5931, 'test': 5952}


In [4]:
# Top-K answers
answer_counts = train_df["answer_norm"].value_counts()
TOP_K_ANSWERS = answer_counts.head(TOP_K).index.tolist()

train_k = train_df[train_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)
val_k = val_df[val_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)
test_k = test_df[test_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)

print("Top-K answers:", len(TOP_K_ANSWERS))
print({"train": len(train_k), "val": len(val_k), "test": len(test_k)})


Top-K answers: 200
{'train': 46598, 'val': 5884, 'test': 5893}


In [5]:
# Helper utilities
def compute_metrics(y_true, y_pred):
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
    }
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return metrics, report


def save_metrics(prefix, split_name, metrics, report, extra=None):
    payload = {"metrics": metrics, "report": report}
    if extra is not None:
        payload["extra"] = extra
    with open(OUT_DIR / f"{prefix}_metrics_{split_name}.json", "w") as f:
        json.dump(payload, f, indent=2)


def save_predictions(prefix, split_name, df, y_pred):
    cols = [c for c in ["img_id", "question", "answer", "answer_norm"] if c in df.columns]
    pred_df = df[cols].copy()
    pred_df["pred"] = y_pred
    pred_df.to_csv(OUT_DIR / f"{prefix}_pred_{split_name}.csv", index=False)


def save_probs(prefix, split_name, probs, classes, ids):
    np.savez(
        OUT_DIR / f"{prefix}_probs_{split_name}.npz",
        probs=probs,
        classes=np.array(classes),
        img_ids=np.array(ids),
    )


def align_probs(probs, from_classes, target_classes):
    idx = {c: i for i, c in enumerate(from_classes)}
    return np.stack([probs[:, idx[c]] for c in target_classes], axis=1)


results = {"text": {}, "image": {}, "fusion_weighted": {}, "fusion_stacking": {}}


In [6]:
# Text-only classifier (TF-IDF + Logistic Regression)
vectorizer = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=NGRAM_RANGE)
X_text_train = vectorizer.fit_transform(train_k["question_norm"])
X_text_val = vectorizer.transform(val_k["question_norm"]) if len(val_k) else None
X_text_test = vectorizer.transform(test_k["question_norm"]) if len(test_k) else None

y_train = train_k["answer_norm"].values
text_clf = LogisticRegression(max_iter=1000, n_jobs=-1)
text_clf.fit(X_text_train, y_train)


def eval_text(df, X, split_name):
    if X is None or len(df) == 0:
        return None, None, None
    y_true = df["answer_norm"].values
    y_pred = text_clf.predict(X)
    probs = text_clf.predict_proba(X)
    metrics, report = compute_metrics(y_true, y_pred)
    save_metrics("text_topk", split_name, metrics, report)
    save_predictions("text_topk", split_name, df, y_pred)
    save_probs("text_topk", split_name, probs, text_clf.classes_, df["img_id"].values)
    print(f"text {split_name}", metrics)
    return y_pred, probs, metrics

text_train_pred, text_train_probs, results["text"]["train"] = eval_text(train_k, X_text_train, "train")
text_val_pred, text_val_probs, results["text"]["val"] = eval_text(val_k, X_text_val, "val")
text_test_pred, text_test_probs, results["text"]["test"] = eval_text(test_k, X_text_test, "test")


text train {'accuracy': 0.6537405038842868, 'macro_f1': 0.04086006780379597}
text val {'accuracy': 0.6526172671651937, 'macro_f1': 0.05755593440380406}
text test {'accuracy': 0.6526387239097234, 'macro_f1': 0.05818740242360812}


In [7]:
# Image-only classifier (frozen ViT embeddings + Logistic Regression)
class ImageDS(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row["img_id"]
        img = Image.open(row["image_path"]).convert("RGB")
        return img_id, img


def collate_fn(batch):
    ids = [b[0] for b in batch]
    images = [b[1] for b in batch]
    inputs = processor(images=images, return_tensors="pt")
    return ids, inputs["pixel_values"]


def compute_embeddings(unique_df):
    ds = ImageDS(unique_df)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)
    emb_map = {}
    with torch.no_grad():
        for ids, pixels in tqdm(dl, desc="ViT embed"):
            pixels = pixels.to(DEVICE)
            out = vit(pixels).last_hidden_state[:, 0, :]
            for i, img_id in enumerate(ids):
                emb_map[img_id] = out[i].cpu().numpy()
    return emb_map


processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
try:
    vit = ViTModel.from_pretrained(MODEL_NAME, torch_dtype=TORCH_DTYPE)
    vit = vit.to(DEVICE)
except RuntimeError as e:
    print(f"Falling back to CPU for ViT (error: {e}).")
    DEVICE = torch.device("cpu")
    TORCH_DTYPE = torch.float32
    vit = ViTModel.from_pretrained(MODEL_NAME, torch_dtype=TORCH_DTYPE).to(DEVICE)
vit.eval()

EMB_PATH = DATA_ROOT / "2_modeling" / "02_image_only" / "out" / "image_embeddings.npz"
unique_imgs = pd.concat([train_k, val_k, test_k])[ ["img_id", "image_path"] ].drop_duplicates()

if EMB_PATH.exists():
    data = np.load(EMB_PATH, allow_pickle=True)
    img_ids = data["img_ids"].tolist()
    embeddings = data["embeddings"]
    emb_map = {img_id: embeddings[i] for i, img_id in enumerate(img_ids)}
    print("Loaded cached embeddings:", len(emb_map))
else:
    emb_map = compute_embeddings(unique_imgs)
    img_ids = list(emb_map.keys())
    embeddings = np.stack([emb_map[i] for i in img_ids])
    EMB_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.savez(EMB_PATH, img_ids=np.array(img_ids), embeddings=embeddings)
    print("Saved embeddings:", EMB_PATH)


def build_X_img(df):
    return np.stack([emb_map[i] for i in df["img_id"].tolist()])

X_img_train = build_X_img(train_k)
X_img_val = build_X_img(val_k) if len(val_k) else None
X_img_test = build_X_img(test_k) if len(test_k) else None

img_scaler = StandardScaler()
X_img_train = img_scaler.fit_transform(X_img_train)
if X_img_val is not None:
    X_img_val = img_scaler.transform(X_img_val)
if X_img_test is not None:
    X_img_test = img_scaler.transform(X_img_test)

img_clf = LogisticRegression(max_iter=1000, n_jobs=-1)
img_clf.fit(X_img_train, train_k["answer_norm"].values)


def eval_image(df, X, split_name):
    if X is None or len(df) == 0:
        return None, None, None
    y_true = df["answer_norm"].values
    y_pred = img_clf.predict(X)
    probs = img_clf.predict_proba(X)
    metrics, report = compute_metrics(y_true, y_pred)
    save_metrics("image_topk", split_name, metrics, report)
    save_predictions("image_topk", split_name, df, y_pred)
    save_probs("image_topk", split_name, probs, img_clf.classes_, df["img_id"].values)
    print(f"image {split_name}", metrics)
    return y_pred, probs, metrics

image_train_pred, image_train_probs, results["image"]["train"] = eval_image(train_k, X_img_train, "train")
image_val_pred, image_val_probs, results["image"]["val"] = eval_image(val_k, X_img_val, "val")
image_test_pred, image_test_probs, results["image"]["test"] = eval_image(test_k, X_img_test, "test")


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


Falling back to CPU for ViT (error: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
).
Loaded cached embeddings: 6500
image train {'accuracy': 0.28117086570239064, 'macro_f1': 0.01605103938825842}
image val {'accuracy': 0.2141400407885792, 'macro_f1': 0.016707112340534645}
image test {'accuracy': 0.2177159341591719, 'macro_f1': 0.017389820326222542}


In [8]:
# Late fusion: weighted averaging of probabilities
best_weight = None
best_val = None

if text_val_probs is None or image_val_probs is None:
    print("Skipping weighted fusion (missing validation probabilities).")
else:
    classes_text = list(text_clf.classes_)
    classes_image = list(img_clf.classes_)
    if set(classes_text) != set(classes_image):
        raise ValueError("Class mismatch between text and image models.")
    # Align probabilities to the same class order
    target_classes = classes_text
    text_val_al = align_probs(text_val_probs, classes_text, target_classes)
    image_val_al = align_probs(image_val_probs, classes_image, target_classes)

    grid_rows = []
    y_val_true = val_k["answer_norm"].values

    for w in WEIGHT_GRID:
        fused = w * text_val_al + (1 - w) * image_val_al
        y_pred = np.array(target_classes)[np.argmax(fused, axis=1)]
        metrics, report = compute_metrics(y_val_true, y_pred)
        grid_rows.append({"weight": float(w), **metrics})
        if (best_val is None) or (metrics["macro_f1"] > best_val["metrics"]["macro_f1"] + 1e-9) or (
            abs(metrics["macro_f1"] - best_val["metrics"]["macro_f1"]) < 1e-9
            and metrics["accuracy"] > best_val["metrics"]["accuracy"]
        ):
            best_val = {
                "weight": float(w),
                "metrics": metrics,
                "report": report,
                "y_pred": y_pred,
                "probs": fused,
            }

    grid_df = pd.DataFrame(grid_rows)
    grid_df.to_csv(OUT_DIR / "weighted_grid_val.csv", index=False)

    save_metrics("fusion_weighted", "val", best_val["metrics"], best_val["report"], extra={"weight": best_val["weight"]})
    save_predictions("fusion_weighted", "val", val_k, best_val["y_pred"])
    save_probs("fusion_weighted", "val", best_val["probs"], target_classes, val_k["img_id"].values)
    results["fusion_weighted"]["val"] = best_val["metrics"]
    best_weight = best_val["weight"]

    if text_test_probs is not None and image_test_probs is not None:
        text_test_al = align_probs(text_test_probs, classes_text, target_classes)
        image_test_al = align_probs(image_test_probs, classes_image, target_classes)
        fused_test = best_weight * text_test_al + (1 - best_weight) * image_test_al
        y_test_true = test_k["answer_norm"].values
        y_test_pred = np.array(target_classes)[np.argmax(fused_test, axis=1)]
        test_metrics, test_report = compute_metrics(y_test_true, y_test_pred)
        save_metrics("fusion_weighted", "test", test_metrics, test_report, extra={"weight": best_weight})
        save_predictions("fusion_weighted", "test", test_k, y_test_pred)
        save_probs("fusion_weighted", "test", fused_test, target_classes, test_k["img_id"].values)
        results["fusion_weighted"]["test"] = test_metrics
        print(f"weighted fusion best w={best_weight:.2f}", "val", best_val["metrics"], "test", test_metrics)


weighted fusion best w=0.45 val {'accuracy': 0.6244051665533651, 'macro_f1': 0.06515573946866435} test {'accuracy': 0.6259969455285932, 'macro_f1': 0.06927784791751368}


In [9]:
# Late fusion: stacking meta-classifier (LogReg on concatenated probs)
stack_val_metrics = None
stack_test_metrics = None

if text_val_probs is None or image_val_probs is None:
    print("Skipping stacking (missing validation probabilities).")
else:
    target_classes = list(text_clf.classes_)
    text_val_al = align_probs(text_val_probs, text_clf.classes_, target_classes)
    image_val_al = align_probs(image_val_probs, img_clf.classes_, target_classes)
    X_meta_train = np.hstack([text_val_al, image_val_al])
    y_meta_train = val_k["answer_norm"].values

    meta_clf = LogisticRegression(max_iter=1000, n_jobs=-1)
    meta_clf.fit(X_meta_train, y_meta_train)

    def eval_meta(df, text_probs, image_probs, split_name):
        if df is None or text_probs is None or image_probs is None or len(df) == 0:
            return None
        X_meta = np.hstack([text_probs, image_probs])
        y_true = df["answer_norm"].values
        y_pred = meta_clf.predict(X_meta)
        probs = meta_clf.predict_proba(X_meta)
        metrics, report = compute_metrics(y_true, y_pred)
        save_metrics("fusion_stacking", split_name, metrics, report)
        save_predictions("fusion_stacking", split_name, df, y_pred)
        save_probs("fusion_stacking", split_name, probs, meta_clf.classes_, df["img_id"].values)
        print(f"stacking {split_name}", metrics)
        return metrics

    stack_val_metrics = eval_meta(val_k, text_val_al, image_val_al, "val")
    results["fusion_stacking"]["val"] = stack_val_metrics

    if text_test_probs is not None and image_test_probs is not None:
        text_test_al = align_probs(text_test_probs, text_clf.classes_, target_classes)
        image_test_al = align_probs(image_test_probs, img_clf.classes_, target_classes)
        stack_test_metrics = eval_meta(test_k, text_test_al, image_test_al, "test")
        results["fusion_stacking"]["test"] = stack_test_metrics


stacking val {'accuracy': 0.7911284840244731, 'macro_f1': 0.10343703931520028}
stacking test {'accuracy': 0.7771932801629051, 'macro_f1': 0.10090927616155686}


In [10]:
# Summary table
summary_rows = []
for model, splits in results.items():
    for split, mets in splits.items():
        if mets is None:
            continue
        summary_rows.append({"model": model, "split": split, "accuracy": mets["accuracy"], "macro_f1": mets["macro_f1"]})

if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    summary_pivot = summary_df.pivot(index="model", columns="split", values=["accuracy", "macro_f1"]).round(4)
    print("Best weight:" , best_weight)
    display(summary_pivot)
else:
    print("No metrics collected.")


Best weight: 0.45


accuracy                 macro_f1                
split               test   train     val     test   train     val
model                                                            
fusion_stacking   0.7772     NaN  0.7911   0.1009     NaN  0.1034
fusion_weighted   0.6260     NaN  0.6244   0.0693     NaN  0.0652
image             0.2177  0.2812  0.2141   0.0174  0.0161  0.0167
text              0.6526  0.6537  0.6526   0.0582  0.0409  0.0576